# 02 — Under the hood: where prompts come from, and what optimizers tune

**What you'll learn**

- Adapters: the machinery that turns `signature + demos + inputs` into an actual prompt.
- `dspy.Example` — the data atom, and its input/label split.
- The keystone of the whole course: every predictor carries exactly **two learnable parameters** — its instructions and its demos.
- Your first optimizer, `LabeledFewShot`, and `program.save()` — the `state_dict()` of DSPy.

*Time: ~25 min. Cost: < $0.02. Cached reruns are free.*

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import dspy
from dotenv import load_dotenv              # pip install -r requirements.txt if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead
# STRONG_MODEL = "openrouter/moonshotai/kimi-k2.6"
# STRONG_MODEL = "openrouter/openai/gpt-4o"

lm = dspy.LM(MODEL, max_tokens=1000)        # cache=True by default: reruns are ~free
dspy.configure(lm=lm)

# Optional tracing for any notebook (taught in notebook 11) — pick ONE option:
# Option A, MLflow: logs land in ./mlflow.db; browse via `mlflow ui` (run from repo root).
#   import mlflow
#   mlflow.set_experiment("dspy-lab")
#   mlflow.dspy.autolog()
# Option B, Arize Phoenix: run `phoenix serve` first; UI at http://localhost:6006.
#   from phoenix.otel import register
#   register(project_name="dspy-lab", auto_instrument=True)

## The question this notebook answers

In notebook 01 you never wrote a prompt, yet `inspect_history` showed a rather elaborate one. So: **who wrote it?**

The answer is a component called an **adapter**. Every LM call in DSPy flows through this pipeline:

```
  signature  +  demos  +  inputs
              |
              v
       Adapter.format()      ->  messages   (the actual prompt)
              |
              v
             LM              ->  raw completion text
              |
              v
       Adapter.parse()       ->  Prediction(field=value, ...)
```

The signature says *what*; the adapter decides *how it is spelled on the wire*. Because the two are separate, the same program can be re-rendered for different models, formats — and, soon, different learned prompts — without touching your code.

You can run the format step by hand. This is the single most demystifying call in DSPy — the prompt, materialized as plain data:

In [ ]:
qa = dspy.Predict("question -> answer")

messages = dspy.ChatAdapter().format(
    signature=qa.signature,
    demos=[],
    inputs={"question": "Why do lighthouses use rotating lights?"},
)
for m in messages:
    print("=" * 20, m["role"], "=" * 20)
    print(m["content"])

> **What you should see (deterministic — no LM call was made):** a system message that lists your input/output fields and demands replies shaped like
>
> ```
> [[ ## answer ## ]]
> {answer}
> ```
>
> followed by a user message containing your question wrapped in `[[ ## question ## ]]`. Those bracketed markers are `ChatAdapter`'s wire format — that is exactly what you saw in `inspect_history` in notebook 01, now explained.

`ChatAdapter` is the default. Others exist — `JSONAdapter` (structured-output/JSON mode, sturdier for typed fields), `XMLAdapter`, `TwoStepAdapter` — and they are swappable without touching the program:

In [ ]:
with dspy.context(adapter=dspy.JSONAdapter()):
    pred = qa(question="Why do lighthouses use rotating lights?")

dspy.inspect_history(n=1)

> **What you should see:** the same *task*, but the request now asks for a JSON object with an `"answer"` key, and the raw reply is JSON.
>
> Same signature, different wire format — the what/how separation from notebook 01, demonstrated. This separation is also *why prompt optimization is safe*: an optimizer can rewrite instructions and attach demos while the adapter guarantees the result is still a well-formed prompt.

## The data atom: `dspy.Example`

Before we meet the learnable parameters, we need DSPy's unit of data. An `Example` is a dict-like bag of named fields with one extra bit of bookkeeping: *which fields are inputs*. Everything else is treated as a label.

In [ ]:
ex = dspy.Example(
    question="What keeps a lighthouse lens from cracking in the cold?",
    answer="brass expansion joints",
).with_inputs("question")

print(ex.inputs())    # what a program would receive
print(ex.labels())    # what only the metric/optimizer may look at

> **What you should see:** `Example({'question': ...})` and `Example({'answer': ...})` — the same object, viewed through its two roles.

Two gotchas worth knowing now: `with_inputs(...)` returns a **copy** (it does not mutate), and calling `.inputs()` on an Example that never got `with_inputs` raises an error — DSPy refuses to guess which fields are inputs. This input/label split is what lets one object serve as a test question, a training point, or — as you are about to see — a few-shot demo.

## The keystone: two parameters per predictor

Here is the model of DSPy to carry for the rest of the course. Every predictor — every `dspy.Predict`, and everything built from it — owns exactly two pieces of learnable state:

```
        dspy.Predict("question -> answer")
       +-----------------------------------+
       |  signature.instructions   <- text |   "how to do the task"
       |  demos: list[Example]     <- data |   "worked examples to imitate"
       +-----------------------------------+
```

θ = (instructions, demos). That's it. The adapter renders both into the prompt on every call. **A DSPy optimizer is any procedure that writes better values into these two slots.** Weight finetuning exists too (notebook 09), but instructions + demos are the parameters every optimizer you'll meet tunes first.

Right now, both slots hold defaults:

In [ ]:
print(repr(qa.signature.instructions))   # auto-generated from the signature
print(qa.demos)                          # empty — no worked examples yet

## The simplest optimizer that could possibly work

`LabeledFewShot` fills the `demos` slot by copying in labeled examples from a trainset. No search, no cleverness, **zero LM calls** — which makes it the perfect first `compile()`:

In [ ]:
trainset = [
    dspy.Example(question="What is the tallest lighthouse in the world?",
                 answer="Jeddah Light").with_inputs("question"),
    dspy.Example(question="What fuel did 18th-century lighthouses burn?",
                 answer="whale oil").with_inputs("question"),
    dspy.Example(question="What is the top room of a lighthouse called?",
                 answer="the lantern room").with_inputs("question"),
    dspy.Example(question="Which empire built the Pharos of Alexandria?",
                 answer="Ptolemaic Egypt").with_inputs("question"),
]

optimizer = dspy.LabeledFewShot(k=3)
qa_fewshot = optimizer.compile(qa, trainset=trainset)

qa_fewshot(question="What does a Fresnel lens do?")

Note the shape of what just happened, because every optimizer in this course repeats it:

```
optimized_program = optimizer.compile(program, trainset=...)
```

`compile` never mutates your program — it returns a **new** one with different parameter values. (`optimizer.step()` updates θ in place; `compile()` is more like training returning a fresh checkpoint.)

In [ ]:
dspy.inspect_history(n=1)

> **What you should see:** the same `[[ ## ... ## ]]` scaffolding as before, but now with three worked question→answer pairs *before* your actual question. The demos slot, rendered by the adapter.

## `save()` — read your parameters with your own eyes

Because θ is just text and examples, a trained DSPy program serializes to *readable JSON*. This file plays the role of `state_dict()`, and reading it is the best debugging habit this course can teach you:

In [ ]:
import json
from pathlib import Path

Path("artifacts").mkdir(exist_ok=True)
qa_fewshot.save("artifacts/02_labeled.json")

state = json.loads(Path("artifacts/02_labeled.json").read_text())
print(json.dumps(state, indent=2)[:1200])

> **What you should see (deterministic):** a JSON object with one entry per predictor (here just `self`), each holding `demos` — your three examples, verbatim — and `signature` with its `instructions`. Plus some metadata about versions.
>
> In every optimizer notebook (05–07) we will come back to this file format and *read what was learned*. When an optimized program behaves oddly, you do not stare at loss curves — you open the JSON and read the prompts. (`program.load(path)` restores the state into a matching program; there is also a whole-program pickle format, covered in notebook 05.)

## The closing trick

One more fact, small but load-bearing: `dspy.Prediction` is a **subclass of `dspy.Example`**.

Think about what that means. A program's *outputs* are the same kind of object as its *training data* — which means an output that your metric approves of can be turned around and inserted into some predictor's `demos` slot as a worked example. A program generating its own training demos, filtered by quality.

That is not a party trick; it is precisely the algorithm of notebook 05. See you there — after we build a program actually worth optimizing (03) and a metric to judge it with (04).

## Recap

| Concept | One-liner |
|---|---|
| adapter | renders `signature + demos + inputs` into messages; parses the reply |
| `ChatAdapter().format(...)` | materialize the exact prompt, no LM call |
| `dspy.JSONAdapter` | same signature, JSON wire format; set via `dspy.configure`/`dspy.context` |
| `dspy.Example` / `with_inputs` | data atom; explicit input-vs-label split; `with_inputs` returns a copy |
| **θ = (instructions, demos)** | the two learnable slots on every predictor |
| `optimizer.compile(program, trainset=...)` | returns a *new* program with better θ |
| `program.save("x.json")` | human-readable learned parameters — always read them |

## Exercises

1. Recompile with `k=1` and print `ChatAdapter().format(qa_fewshot.signature, qa_fewshot.demos, {"question": "..."})` for both versions. Diff the prompts by eye.
2. Build `qa_strict = qa.deepcopy()` and set `qa_strict.signature = qa.signature.with_instructions("Answer with a noun phrase, no verbs.")`. Save both programs and diff the two JSON files — which of the two θ slots changed?
3. Predict what `JSONAdapter` does with demos (how would worked examples appear in a JSON-mode conversation?), then check with `dspy.context(adapter=...)` + `inspect_history`.

**Next up:** composition — wiring predictors and plain Python into a real multi-step program, over a world the model has never read about.